In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

# ==========================================================
# BRONZE TO SILVER - DATABRICKS VERSION
# Source orders are nested: one row per order, items is an array.
# Silver grain: one row per order item (order_id + product_id)
# ==========================================================

pipeline_run_id = str(uuid.uuid4())

# ----------------------------------------------------------
# 1. Read Bronze tables
# ----------------------------------------------------------
orders_df = spark.table("ecom_medallion_arc_pipeline.bronze.orders_bronze")
products_df = spark.table("ecom_medallion_arc_pipeline.bronze.products_bronze")

# Uncomment to confirm the schema if you hit another column error
orders_df.printSchema()

def blank_as_null(column_name):
    return F.when(F.trim(F.col(column_name)) == "", None).otherwise(F.col(column_name))

VALID_STATUSES = ["delivered", "shipped", "processing", "cancelled", "returned"]

# ----------------------------------------------------------
# 2. Explode items array -> one row per order item
# explode_outer keeps orders with null/empty items so they get rejected
# instead of silently disappearing
# ----------------------------------------------------------
orders_exploded_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        F.col("status").alias("order_status"),
        "payment_method",
        F.col("shipping_address.city").alias("ship_city"),
        F.col("shipping_address.state").alias("ship_state"),
        "load_date",
        "source_file_name",
        "ingestion_timestamp",
        F.explode_outer("items").alias("item")
    )
    .withColumn("product_id", F.col("item.product_id"))
    .withColumn("quantity", F.col("item.quantity"))
    .withColumn("unit_price", F.col("item.unit_price"))
    .withColumn("discount_pct", F.col("item.discount_pct"))
)

# ----------------------------------------------------------
# 3. Standardize before validation
# ----------------------------------------------------------
orders_prepared_df = (
    orders_exploded_df
    .withColumn("order_id", blank_as_null("order_id"))
    .withColumn("customer_id", blank_as_null("customer_id"))
    .withColumn("product_id", blank_as_null("product_id"))
    .withColumn("order_status_std", F.lower(F.trim(F.col("order_status"))))
    .withColumn("quantity_num", F.col("quantity").cast("int"))
    .withColumn("unit_price_num", F.col("unit_price").cast("double"))
    .withColumn("discount_pct_num", F.coalesce(F.col("discount_pct").cast("double"), F.lit(0.0)))
    .withColumn(
        "order_timestamp",
        F.coalesce(
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd'T'HH:mm:ss"))
        )
    )
)

# ----------------------------------------------------------
# 4. Duplicate detection at item grain (order_id + product_id)
# Keep first occurrence, reject later duplicates
# ----------------------------------------------------------
order_dup_window = Window.partitionBy("order_id", "product_id").orderBy(
    F.col("load_date").asc(),
    F.col("source_file_name").asc(),
    F.col("ingestion_timestamp").asc()
)

orders_with_dup_flag_df = orders_prepared_df.withColumn(
    "dup_rank", F.row_number().over(order_dup_window)
)

# ----------------------------------------------------------
# 5. Data quality rules
# ----------------------------------------------------------
orders_validated_df = (
    orders_with_dup_flag_df
    .withColumn(
        "rejection_reason",
        F.when(F.col("order_id").isNull(), F.lit("NULL_ORDER_ID"))
         .when(F.col("customer_id").isNull(), F.lit("NULL_CUSTOMER_ID"))
         .when(F.col("product_id").isNull(), F.lit("NULL_PRODUCT_ID"))
         .when(F.col("order_timestamp").isNull(), F.lit("INVALID_ORDER_DATE"))
         .when(F.col("quantity_num").isNull(), F.lit("NULL_OR_INVALID_QUANTITY"))
         .when(F.col("quantity_num") <= 0, F.lit("INVALID_QUANTITY"))
         .when(F.col("unit_price_num").isNull(), F.lit("NULL_OR_INVALID_UNIT_PRICE"))
         .when(F.col("unit_price_num") <= 0, F.lit("INVALID_UNIT_PRICE"))
         .when((F.col("discount_pct_num") < 0) | (F.col("discount_pct_num") > 100), F.lit("INVALID_DISCOUNT"))
         .when(~F.col("order_status_std").isin(*VALID_STATUSES), F.lit("INVALID_ORDER_STATUS"))
         .when(F.col("dup_rank") > 1, F.lit("DUPLICATE_ORDER_ITEM"))
         .otherwise(F.lit(None))
    )
)

# ----------------------------------------------------------
# 6. Split rejected and valid
# ----------------------------------------------------------
rejected_orders_df = (
    orders_validated_df
    .filter(F.col("rejection_reason").isNotNull())
    .select(
        F.to_json(
            F.struct(
                "order_id", "customer_id", "product_id", "order_date",
                "quantity", "unit_price", "discount_pct", "order_status"
            )
        ).alias("raw_record"),
        "rejection_reason",
        "source_file_name",
        "load_date",
        F.lit(pipeline_run_id).alias("pipeline_run_id")
    )
)

valid_orders_df = orders_validated_df.filter(F.col("rejection_reason").isNull())

# ----------------------------------------------------------
# 7. Clean and dedup product master
# ----------------------------------------------------------
products_prepared_df = (
    products_df
    .withColumn("product_id", blank_as_null("product_id"))
    .withColumn("product_name", blank_as_null("product_name"))
    .withColumn("category", blank_as_null("category"))
    .withColumn("brand", blank_as_null("brand"))
)

products_valid_df = (
    products_prepared_df
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("product_name").isNotNull())
    .filter(F.col("category").isNotNull())
    .filter(F.col("brand").isNotNull())
)

product_dup_window = Window.partitionBy("product_id").orderBy(
    F.col("load_date").asc(),
    F.col("source_file_name").asc(),
    F.col("ingestion_timestamp").asc()
)

products_dedup_df = (
    products_valid_df
    .withColumn("prod_rank", F.row_number().over(product_dup_window))
    .filter(F.col("prod_rank") == 1)
    .select(
        F.col("product_id").alias("p_product_id"),
        "product_name", "category", "brand"
    )
)

# ----------------------------------------------------------
# 8. Enrich valid orders with product details
# ----------------------------------------------------------
gross = F.col("o.quantity_num") * F.col("o.unit_price_num")

orders_curated_df = (
    valid_orders_df.alias("o")
    .join(
        products_dedup_df.alias("p"),
        F.col("o.product_id") == F.col("p.p_product_id"),
        "left"
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.product_id").alias("product_id"),
        F.coalesce(F.col("p.product_name"), F.lit("UNKNOWN")).alias("product_name"),
        F.coalesce(F.col("p.category"), F.lit("UNKNOWN")).alias("category"),
        F.coalesce(F.col("p.brand"), F.lit("UNKNOWN")).alias("brand"),
        F.col("o.order_timestamp").alias("order_timestamp"),
        F.to_date(F.col("o.order_timestamp")).alias("order_date"),
        F.col("o.quantity_num").alias("quantity"),
        F.col("o.unit_price_num").alias("unit_price"),
        F.col("o.discount_pct_num").alias("discount_pct"),
        gross.cast("double").alias("gross_amount"),
        F.round(gross * (1 - F.col("o.discount_pct_num") / 100), 2).alias("order_amount"),
        F.col("o.order_status_std").alias("order_status"),
        F.col("o.payment_method").alias("payment_method"),
        F.col("o.ship_city").alias("ship_city"),
        F.col("o.ship_state").alias("ship_state"),
        F.col("o.load_date").alias("load_date"),
        F.lit(pipeline_run_id).alias("pipeline_run_id")
    )
)

# ----------------------------------------------------------
# 9. Write Silver tables
# ----------------------------------------------------------
(orders_curated_df.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("ecom_medallion_arc_pipeline.silver.orders_curated"))

(rejected_orders_df.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("ecom_medallion_arc_pipeline.silver.rejected_orders"))

# ----------------------------------------------------------
# 10. Validation
# ----------------------------------------------------------
print("Pipeline Run ID:", pipeline_run_id)

print("Curated Orders Count:")
spark.sql("SELECT COUNT(*) AS total_rows FROM ecom_medallion_arc_pipeline.silver.orders_curated").show()

print("Rejected Orders Count:")
spark.sql("SELECT COUNT(*) AS total_rows FROM ecom_medallion_arc_pipeline.silver.rejected_orders").show()

print("Rejection Summary:")
spark.sql("""
SELECT rejection_reason, COUNT(*) AS cnt
FROM ecom_medallion_arc_pipeline.silver.rejected_orders
GROUP BY rejection_reason
ORDER BY cnt DESC
""").show()

print("Products not matched in master (should be 0 for clean data):")
spark.sql("""
SELECT COUNT(*) AS unknown_products
FROM ecom_medallion_arc_pipeline.silver.orders_curated
WHERE product_name = 'UNKNOWN'
""").show()

display(spark.table("ecom_medallion_arc_pipeline.silver.orders_curated"))
display(spark.table("ecom_medallion_arc_pipeline.silver.rejected_orders"))